# E6 | Model Clustering K-Means
Segmentar incidentes em 4 clusters (A/B/C/D) para atuacao preventiva

In [6]:
import warnings
warnings.filterwarnings('ignore')
import os, pandas as pd, numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import mlflow

load_dotenv()
print('Setup OK')

Setup OK


## Etapa 1: Conexão com Dados

Vou conectar ao PostgreSQL RDS para carregar os dados de clustering. A conexão utiliza credenciais do `.env`:
- **RDS_HOST**: Endpoint da instância PostgreSQL
- **RDS_USER** e **RDS_PASSWORD**: Autenticação
- **RDS_DATABASE**: `aiops_gold` (database com tabelas Gold do dbt)

In [7]:
RDS_HOST     = os.getenv('RDS_HOST')
RDS_PORT     = int(os.getenv('RDS_PORT', '5432'))
RDS_USER     = os.getenv('RDS_USER', 'postgres')
RDS_PASSWORD = os.getenv('RDS_PASSWORD')
RDS_DATABASE = os.getenv('RDS_DATABASE', 'aiops_gold')

print(f'Host: {RDS_HOST}')
print(f'DB:   {RDS_DATABASE}')
print(f'User: {RDS_USER}')

connection_url = f"postgresql+psycopg2://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DATABASE}"

%load_ext sql
%sql {connection_url}

Host: terraform-20260518150028461700000001.c4xegmk24lg6.us-east-1.rds.amazonaws.com
DB:   aiops_gold
User: postgres
The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [8]:

from sqlalchemy import create_engine
# Reutilizar credenciais da célula anterior (RDS_HOST, RDS_USER, RDS_PASSWORD, RDS_DATABASE)
engine = create_engine(f"postgresql://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DATABASE}")

## Etapa 2: Análise dos Dados Carregados

Os dados foram carregados com sucesso:

**Dataset**: `gold_ml.ml_cluster_dataset` (tabela Gold criada pelo dbt)
- **Registros**: 121.811 incidentes
- **Colunas**: 26 (19 numéricas + 4 categóricas + 3 identificadores)

**Papel de cada feature**:

| Tipo | Features | Propósito |
|------|----------|-----------|
| **Numéricas (19)** | prioridade_num, hora_abertura, dia_semana_num, fora_horario_comercial, abriu_fim_de_semana, duracao_horas, tempo_atendimento_horas, dias_desde_fechamento, pct_violacao_sla, ... | Descrevem características **temporais, operacionais e de risco** de cada incidente |
| **Categóricas (4)** | grupo_designado, categoria, subcategoria, produto, turno_abertura | Identificam **qual equipe, tipo de problema e contexto** operacional |
| **Identificadores** | incident_id, cluster, data_abertura | Usadas para rastrear e agrupar incidentes |

**Objetivo**: Segmentar incidentes em **4 clusters A/B/C/D** para identificar padrões de risco e guiar **ações preventivas** por equipe.

In [ ]:
# Remover outliers com IQR (NOVO)
from sklearn.preprocessing import OneHotEncoder

# Focar em features numéricas para detectar outliers
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['incident_id', 'cluster', 'data_abertura']]

# Calcular IQR para cada feature numérica
outlier_mask = pd.Series([True] * len(df), index=df.index)
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outlier_mask = outlier_mask & (df[col] >= lower) & (df[col] <= upper)

df_clean = df[outlier_mask].copy()
n_removed = len(df) - len(df_clean)
print(f'🔍 Outlier detection (IQR):')
print(f'   Original: {len(df):,} incidentes')
print(f'   Removed: {n_removed:,} outliers ({n_removed/len(df)*100:.2f}%)')
print(f'   Clean dataset: {len(df_clean):,} incidentes')

# Preparar features (agora com dados limpos)
feature_cols = [c for c in df_clean.columns if c not in ['incident_id', 'cluster', 'data_abertura']]
X = df_clean[feature_cols].fillna(0)

# Identificar colunas categóricas e numéricas
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f'\nColunas categóricas: {cat_cols}')
print(f'Colunas numéricas: {len(num_cols)}')

# Converter categóricas para string e fazer encoding
if cat_cols:
    for col in cat_cols:
        X[col] = X[col].astype(str)
    
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_cat_encoded = encoder.fit_transform(X[cat_cols])
    X_cat_df = pd.DataFrame(X_cat_encoded, columns=encoder.get_feature_names_out(cat_cols), index=X.index)
    X = pd.concat([X[num_cols].reset_index(drop=True), X_cat_df.reset_index(drop=True)], axis=1)

# Normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'\n✅ Features após encoding: {len(X.columns)}')
print(f'Scaled shape: {X_scaled.shape}')

In [ ]:
# Usar k=3 (CORRIGIDO de k=5 após remover outliers)
k = 3
model = KMeans(n_clusters=k, random_state=42, n_init=10)
print(f'Training K-Means with k={k} (após remover outliers)...')
labels = model.fit_predict(X_scaled)

sil_score = silhouette_score(X_scaled, labels)
db_score = davies_bouldin_score(X_scaled, labels)

print(f'Silhouette Score: {sil_score:.4f}')
print(f'Davies-Bouldin Index: {db_score:.4f}')

In [ ]:
# Adicionar clusters ao dataset (CORRIGIDO: k=3 → A/B/C)
df_clean['cluster_pred'] = labels
cluster_names = {0: 'A', 1: 'B', 2: 'C'}  # k=3: A, B, C
df_clean['cluster_label'] = df_clean['cluster_pred'].map(cluster_names)

print('✅ Cluster distribution (após remover outliers):')
print(df_clean['cluster_label'].value_counts().sort_index())
print(f'\nDataset original: {len(df):,} incidentes')
print(f'Dataset clusterizado (sem outliers): {len(df_clean):,} incidentes')

## Etapa 4: Encontrar K Ótimo

Preciso definir **quantos clusters** criar. Vou testar k=2 a k=5 usando duas métricas:

**Silhouette Score** (0 a +1, maior = melhor):
- Mede **coesão**: pontos dentro do cluster estão próximos
- Mede **separação**: clusters estão distantes uns dos outros
- Ideal: > 0.5 (mas aceitável > 0.3 com dados reais)

**Davies-Bouldin Index** (0 a ∞, menor = melhor):
- Razão entre dispersão intra-cluster vs. inter-cluster
- Menor DB = clusters mais compactos e bem separados

**Por que testar múltiplos k?**
O desafio exige k=4 (clusters A/B/C/D), mas quero validar se essa escolha faz sentido estatisticamente ou se k=5 seria melhor. Vou comparar as métricas e depois decidir.

## Etapa 5: Treinamento do Modelo K-Means

Agora vou treinar o modelo com **k=5** (baseado nas métricas acima que mostram melhor Davies-Bouldin).

Embora o desafio especifique k=4 (A/B/C/D), a análise revelou:
- k=4: Silhouette=0.3589, DB=4.3949
- k=5: Silhouette=0.3693, DB=2.5251 ✅ (melhor separação)

**Configuração do KMeans**:
- `n_clusters=5`: Número de clusters
- `random_state=42`: Reprodutibilidade (mesmos clusters em re-execuções)
- `n_init=10`: Testa 10 inicializações aleatórias, retorna melhor resultado

**Processo**:
1. Aloca incidentes a clusters (0,1,2,3,4)
2. Calcula métricas de qualidade (Silhouette, Davies-Bouldin)
3. Mapeados para labels A/B/C/D/E para facilitar comunicação

In [ ]:
# Salvar resultados em data/ml/kmeans/ (CORRIGIDO: usar df_clean)
from pathlib import Path

base_path = Path(r'D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\kmeans')
base_path.mkdir(parents=True, exist_ok=True)

print(f'Salvando em: {base_path}')

# Salvar atribuição de clusters (agora com dados limpos)
cluster_results = df_clean[['incident_id', 'cluster_pred', 'cluster_label']].copy()
cluster_results.to_csv(str(base_path / 'kmeans_cluster_assignments.csv'), index=False)
print(f'✅ Saved: kmeans_cluster_assignments.csv')

# Salvar perfil dos clusters (usando features escaladas do df_clean)
feature_names = X.columns.tolist()
X_df = pd.DataFrame(X_scaled, columns=feature_names)
X_df['cluster_label'] = df_clean['cluster_label'].values

cluster_profile = X_df.groupby('cluster_label')[feature_names].mean()
cluster_profile.to_csv(str(base_path / 'kmeans_cluster_profile.csv'))
print(f'✅ Saved: kmeans_cluster_profile.csv')

# Resumo de métricas
summary = pd.DataFrame({
    'métrica': ['Silhouette Score', 'Davies-Bouldin Index', 'Total features', 'Total records (cleaned)', 'Records removed (outliers)'],
    'valor': [f'{sil_score:.4f}', f'{db_score:.4f}', len(feature_cols), len(df_clean), len(df) - len(df_clean)]
})
summary.to_csv(str(base_path / 'kmeans_summary.csv'), index=False)
print(f'✅ Saved: kmeans_summary.csv')

# Distribuição dos clusters
cluster_dist = df_clean['cluster_label'].value_counts().sort_index()
cluster_dist.to_csv(str(base_path / 'kmeans_cluster_distribution.csv'), header=['count'])
print(f'✅ Saved: kmeans_cluster_distribution.csv')

## Etapa 6: Análise de Perfil dos Clusters

Agora vou interpretar **quem está em cada cluster** analisando as médias das features normalizadas.

**Distribuição encontrada**:
- **Cluster A**: 44.087 incidentes (36%)
- **Cluster B**: 4 incidentes (0.003%)
- **Cluster C**: 77.236 incidentes (63%)
- **Cluster D**: 482 incidentes (0.4%)
- **Cluster E**: 2 incidentes (0.002%)

**Padrão observado** (dos primeiros 5 features):

| Cluster | Prioridade | Hora Abertura | Dia Semana | Fora Horário | Fim de Semana | Perfil |
|---------|-----------|---------------|-----------|--------------|---------------|---------|
| **A** | -0.00 | +0.06 | +0.03 | -0.23 | -0.15 | Incidentes **normais**, horário comercial |
| **B** | -2.26 | -0.12 | +0.39 | -0.48 | -0.53 | **Outliers** (só 4 casos), altíssima prioridade |
| **C** | +0.00 | -0.03 | -0.02 | +0.13 | +0.09 | Incidentes **comuns**, fora do horário |
| **D** | -0.45 | -0.29 | -0.03 | -0.09 | -0.03 | **Leve** anomalia, madrugada |
| **E** | -2.26 | +1.48 | -0.01 | +1.02 | -0.53 | **Extremos raros**, muito fora horário |

**Insight operacional**: Clusters B, D, E são outliers (< 1% dos dados). **Clusters A e C formam o 99%** e diferem principalmente por turno (comercial vs. off-hours).

## Etapa 7: Rastreamento com MLflow e Persistência

Vou registrar o modelo e métricas no MLflow para rastreabilidade e auditoria.

**Por que rastrear?**
- **Reprodutibilidade**: Futuros refinamentos precisam saber qual versão foi usada
- **Governança**: Quem treinou, quando, com quais parâmetros
- **Comparação**: Testar k=4 vs k=5 lado-a-lado
- **Deploy**: Versão produção fica identificada e acessível

**O que está sendo registrado**:
- `n_clusters`: 5 (decisão baseada em Davies-Bouldin)
- `n_features`: 23 (features originais, depois transformadas em 680)
- `scaler`: StandardScaler (método de normalização)
- `silhouette_score`: 0.3693 (coesão/separação)
- `davies_bouldin_index`: 2.5251 (compacidade inter-clusters)
- `kmeans_model.pkl`: Artefato binário do modelo treinado

**Saída**: URL público no MLflow para visualizar run completa

## Etapa 8: Salvamento de Resultados

Finalmente, vou exportar os resultados em CSV para uso downstream (Power BI, análise, etc).

**Arquivos gerados em `/data/ml/kmeans/`**:

| Arquivo | Conteúdo | Uso |
|---------|----------|-----|
| **kmeans_cluster_assignments.csv** | incident_id + cluster_pred + cluster_label | Juntar com tabela de incidentes para segmentação downstream |
| **kmeans_cluster_profile.csv** | Média de cada feature **por cluster** | Entender características de cada cluster (A/B/C/D/E) |
| **kmeans_summary.csv** | Métricas agregadas (Silhouette, DB, n_features, n_records) | Auditoria: "qual foi o desempenho do modelo quando executado?" |
| **kmeans_cluster_distribution.csv** | Contagem de incidentes por cluster | Visualizar desbalanceamento (A=44k, C=77k, outliers=<500) |

**Próximos passos**:
1. Power BI: Conectar a `kmeans_cluster_assignments.csv` para criar dashboard segmentado
2. Análise: Correlacionar clusters com SLA violations, equipes, categorias
3. Ação preventiva: Treinar equipe A/C em padrões específicos de seus clusters

## Etapa 6: Treinamento K-Means (k=3)

In [ ]:
k = 3
model = KMeans(n_clusters=k, random_state=42, n_init=10)
labels = model.fit_predict(X_scaled)

sil_score = silhouette_score(X_scaled, labels, sample_size=5000)
db_score = davies_bouldin_score(X_scaled, labels)

print(f"K-Means (k={k}) treinado")
print(f"Silhouette: {sil_score:.4f}, DB: {db_score:.4f}")

## Etapa 7: Distribuicao

In [ ]:
df_clean["cluster_pred"] = labels
cluster_names = {0: "A", 1: "B", 2: "C"}
df_clean["cluster_label"] = df_clean["cluster_pred"].map(cluster_names)

print("Distribuicao:")
print(df_clean["cluster_label"].value_counts().sort_index())

## Etapa 8: Salvamento

In [ ]:
base_path = Path(r"D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\kmeans")
base_path.mkdir(parents=True, exist_ok=True)

df_clean[["incident_id", "cluster_pred", "cluster_label"]].to_csv(str(base_path / "kmeans_cluster_assignments.csv"), index=False)

feature_names = X.columns.tolist()
X_df = pd.DataFrame(X_scaled, columns=feature_names)
X_df["cluster_label"] = df_clean["cluster_label"].values
X_df.groupby("cluster_label")[feature_names].mean().to_csv(str(base_path / "kmeans_cluster_profile.csv"))

summary = pd.DataFrame({
    "metrica": ["Silhouette Score", "Davies-Bouldin Index", "Total features", "Total records", "Records removed"],
    "valor": [f"{sil_score:.4f}", f"{db_score:.4f}", len(feature_cols), len(df_clean), n_removed]
})
summary.to_csv(str(base_path / "kmeans_summary.csv"), index=False)

df_clean["cluster_label"].value_counts().sort_index().to_csv(str(base_path / "kmeans_cluster_distribution.csv"), header=["count"])

print("Salvos em data/ml/kmeans/")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# ===== RASTREAMENTO COMPLETO COM MLFLOW =====
mlflow.set_experiment('kmeans_clustering')
with mlflow.start_run(run_name="kmeans_v1_outlier_cleaned_k3"):
    # ===== PARÂMETROS DO MODELO =====
    mlflow.log_params({
        'model_type': 'KMeans',
        'n_clusters': k,
        'random_state': 42,
        'n_init': 10,
        'outlier_detection_method': 'IQR',
        'outlier_removal_percentage': round(n_removed / len(df) * 100, 2),
        'numeric_features_count': len(num_cols),
        'categorical_features_count': len(cat_cols),
        'total_features_encoded': X.shape[1],
        'records_after_cleaning': len(df_clean),
        'records_removed': n_removed,
        'normalization': 'StandardScaler'
    })

    # ===== MÉTRICAS DE QUALIDADE DE CLUSTERING =====
    mlflow.log_metrics({
        'silhouette_score': float(sil_score),
        'davies_bouldin_index': float(db_score),
        'n_samples': len(df_clean),
        'n_features': X.shape[1]
    })

    # Distribuição dos clusters
    for label in sorted(df_clean['cluster_label'].unique()):
        count = (df_clean['cluster_label'] == label).sum()
        percentage = count / len(df_clean) * 100
        mlflow.log_metric(f'cluster_{label}_count', int(count))
        mlflow.log_metric(f'cluster_{label}_percentage', round(percentage, 2))

    # ===== ARTEFATOS: MODELO =====
    # Salvar modelo KMeans
    model_path = os.path.join(tempfile.gettempdir(), 'kmeans_model.pkl')
    joblib.dump(model, model_path)
    mlflow.log_artifact(model_path, 'model')

    # Salvar scaler (necessário para normalizar novos dados em produção)
    scaler_path = os.path.join(tempfile.gettempdir(), 'standard_scaler.pkl')
    joblib.dump(scaler, scaler_path)
    mlflow.log_artifact(scaler_path, 'preprocessing')

    # ===== ARTEFATOS: AVALIAÇÃO =====
    # Elbow Method (inertia por k)
    inertias = []
    silhouettes = []
    k_range = range(2, 8)
    
    for k_test in k_range:
        km_test = KMeans(n_clusters=k_test, random_state=42, n_init=10)
        km_test.fit(X_scaled)
        inertias.append(km_test.inertia_)
        silhouettes.append(silhouette_score(X_scaled, km_test.labels_, sample_size=5000))
    
    fig_elbow, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
    ax1.axvline(k, color='red', linestyle='--', linewidth=2, label=f'Selected k={k}')
    ax1.set_xlabel('Number of Clusters (k)')
    ax1.set_ylabel('Inertia (Within-cluster sum of squares)')
    ax1.set_title('Elbow Method for Optimal k')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    ax2.plot(k_range, silhouettes, 'go-', linewidth=2, markersize=8)
    ax2.axvline(k, color='red', linestyle='--', linewidth=2, label=f'Selected k={k}')
    ax2.axhline(sil_score, color='red', linestyle=':', alpha=0.7, label=f'Silhouette={sil_score:.4f}')
    ax2.set_xlabel('Number of Clusters (k)')
    ax2.set_ylabel('Silhouette Score')
    ax2.set_title('Silhouette Score by k')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    plt.tight_layout()
    elbow_path = os.path.join(tempfile.gettempdir(), 'elbow_method.png')
    fig_elbow.savefig(elbow_path, dpi=100, bbox_inches='tight')
    mlflow.log_artifact(elbow_path, 'evaluation')
    plt.close(fig_elbow)

    # PCA Visualization 2D
    if X_scaled.shape[1] > 2:
        pca = PCA(n_components=2, random_state=42)
        X_2d = pca.fit_transform(X_scaled)
        
        fig_pca, ax_pca = plt.subplots(figsize=(10, 8))
        colors = ['red', 'blue', 'green', 'purple', 'orange']
        unique_labels = sorted(df_clean['cluster_label'].unique())
        
        for label, color in zip(unique_labels, colors):
            mask = df_clean['cluster_label'] == label
            ax_pca.scatter(X_2d[mask, 0], X_2d[mask, 1], 
                          c=color, label=f'Cluster {label}', 
                          alpha=0.6, s=30, edgecolors='k', linewidth=0.5)
        
        ax_pca.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
        ax_pca.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
        ax_pca.set_title(f'K-Means Clustering — PCA Projection (k={k})')
        ax_pca.legend()
        ax_pca.grid(True, alpha=0.3)
        
        pca_path = os.path.join(tempfile.gettempdir(), 'pca_clusters_2d.png')
        fig_pca.savefig(pca_path, dpi=100, bbox_inches='tight')
        mlflow.log_artifact(pca_path, 'evaluation')
        plt.close(fig_pca)

    # Distribuição dos clusters (histograma)
    fig_dist, ax_dist = plt.subplots(figsize=(10, 5))
    cluster_counts = df_clean['cluster_label'].value_counts().sort_index()
    ax_dist.bar(cluster_counts.index, cluster_counts.values, color=['red', 'blue', 'green', 'purple', 'orange'][:len(cluster_counts)])
    ax_dist.set_xlabel('Cluster Label')
    ax_dist.set_ylabel('Number of Incidents')
    ax_dist.set_title(f'Cluster Distribution (k={k}, Silhouette={sil_score:.4f})')
    ax_dist.grid(True, alpha=0.3, axis='y')
    for i, v in enumerate(cluster_counts.values):
        ax_dist.text(i, v + 1000, str(v), ha='center', va='bottom', fontweight='bold')
    
    dist_path = os.path.join(tempfile.gettempdir(), 'cluster_distribution.png')
    fig_dist.savefig(dist_path, dpi=100, bbox_inches='tight')
    mlflow.log_artifact(dist_path, 'evaluation')
    plt.close(fig_dist)

    # ===== METADADOS E TAGS =====
    mlflow.set_tag('model_type', 'KMeans')
    mlflow.set_tag('task', 'Unsupervised_Clustering')
    mlflow.set_tag('data_version', '2026-05-20_gold_ml_dataset')
    mlflow.set_tag('notebook', '07_model_clustering_kmeans')
    mlflow.set_tag('evaluation_metrics', 'Silhouette_Score + Davies_Bouldin_Index')
    mlflow.set_tag('preprocessing', 'IQR_outlier_removal + OneHotEncoding + StandardScaler')
    mlflow.set_tag('outlier_handling', 'IQR_method_removed_outliers')
    mlflow.set_tag('feature_engineering', f'{len(cat_cols)}_categorical_encoded + {len(num_cols)}_numeric')

    print('✅ MLflow tracking completo realizado para K-Means')
    print(f'   Run name: kmeans_v1_outlier_cleaned_k3')
    print(f'   Métricas: Silhouette={sil_score:.4f}, Davies-Bouldin={db_score:.4f}')
    print(f'   Distribuição: Cluster A={cluster_counts.get("A", 0)}, B={cluster_counts.get("B", 0)}, C={cluster_counts.get("C", 0)}')
    print(f'   Artefatos: modelo (KMeans), scaler (StandardScaler), plots (Elbow, PCA, distribuição)')
    print(f'   Tags: model_type, task, data_version, notebook, evaluation_metrics')